# 4. High-Need Neighborhoods

This notebook rebuilds the story's central finding from the travel-time file and the Census block groups:

1. **The 265 high-need hexagons**, and the 21,057 households (about 5,700 of them without a car) who live in them.
2. **The 39 neighborhoods** they form, where touching hexagons join into one outlined area.
3. **How much the two lines matter**: the count under nine combinations of the bus line and the car-ownership line, and at midday.
4. **The USDA's low-access tracts against the high-need hexagons**: where the two maps agree and where they part.

**How to read this notebook:** Each grey box runs a small piece of Python and prints what it found right below it. A box that starts with `assert` is a check: if any number disagrees with what the site publishes, the notebook stops with an error. Each check is written out in words just above it.

In [1]:
import json
import math
from collections import Counter
from pathlib import Path

snapshots_folder = Path('..') / 'snapshots'


def count_of(items):
    return len(items)


def rounded_to_nearest_hundred(number):
    return round(number, -2)

travel_surface = json.loads((snapshots_folder / 'travel-surface@2026-10-03.json').read_text())
block_group_layer = json.loads((snapshots_folder / 'Quantile_Vehicle@2026-10-03.geojson').read_text())

hexagons = travel_surface['cells']
lived_in_hexagons = [hexagon for hexagon in hexagons if hexagon['populated']]
carless_percent_by_block_group = {
    feature['properties']['GEOID']: feature['properties']['PercentNoVehicle']
    for feature in block_group_layer['features']
    if feature['properties']['Total_Occupied_Housing_Units']
}

## 1. The 265 high-need hexagons

A lived-in hexagon is flagged as **high-need** when it meets both conditions:

1. **Bus trip of 25+ minutes (or no bus service):** The fastest morning rush-hour bus trip to a supermarket rounds to **25 minutes or more**, or no bus reaches one. (We round half-minutes up with `math.floor(minutes + 0.5)` to match the whole-minute numbers on the map's trip cards.)
2. **More than 15% carless households:** Over **15%** of households in its Census block group do not have a car.

In [2]:
def whole_minutes(minutes):
    return math.floor(minutes + 0.5)


def drive_minutes_to_supermarket(hexagon):
    return hexagon['layers']['supermarket']['car']['minutes']


def bus_minutes_to_supermarket(hexagon, period='peak'):
    bus_trip = hexagon['layers']['supermarket']['transit'][period]
    return bus_trip['minutes'] if bus_trip else None


def carless_percent_of(hexagon):
    return carless_percent_by_block_group.get(hexagon['geoid'], 0)


def is_high_need(hexagon, bus_line_minutes=25, carless_line_percent=15, period='peak'):
    if not hexagon['populated']:
        return False
    if carless_percent_of(hexagon) <= carless_line_percent:
        return False
    bus_minutes = bus_minutes_to_supermarket(hexagon, period)
    no_bus_reaches_a_supermarket = bus_minutes is None
    return no_bus_reaches_a_supermarket or whole_minutes(bus_minutes) >= bus_line_minutes


print(f'24.49 minutes counts as {whole_minutes(24.49)}, 24.5 as {whole_minutes(24.5)}')

24.49 minutes counts as 24, 24.5 as 25


In [3]:
high_need_hexagons = [hexagon for hexagon in lived_in_hexagons if is_high_need(hexagon)]
high_need_with_no_bus_trip = [hexagon for hexagon in high_need_hexagons if bus_minutes_to_supermarket(hexagon) is None]

high_need_households = sum(hexagon['households'] for hexagon in high_need_hexagons)
high_need_carless_households = sum(hexagon['households'] * carless_percent_of(hexagon) / 100 for hexagon in high_need_hexagons)

print(f'{count_of(high_need_hexagons)} high-need hexagons:')
print(f'  {count_of(high_need_hexagons) - count_of(high_need_with_no_bus_trip)} with a bus trip of 25 minutes or more')
print(f'  {count_of(high_need_with_no_bus_trip)} with no bus trip at all')
print(f'{high_need_households:,.0f} households live in them, {high_need_carless_households:,.0f} of them without a car')

265 high-need hexagons:
  235 with a bus trip of 25 minutes or more
  30 with no bus trip at all
21,057 households live in them, 5,684 of them without a car


The story rounds these to *about 21,000 households* and *roughly 5,700 carless families*.

**Check:** 265 high-need hexagons, 30 of them with no bus trip; 21,057 households, about 5,700 of them without a car.

In [4]:
assert count_of(high_need_hexagons) == 265
assert count_of(high_need_with_no_bus_trip) == 30
assert round(high_need_households) == 21_057
assert rounded_to_nearest_hundred(high_need_carless_households) == 5_700

## 2. The 39 neighborhoods

On the map, touching high-need hexagons are grouped into contiguous **neighborhood clusters**. Starting from each unvisited high-need hexagon, we step outward to all touching high-need neighbors until the whole cluster is collected, forming **39 neighborhoods** across the county:

In [5]:
def neighbors(row, column):
    columns_in_next_rows = (column, column + 1) if row % 2 == 1 else (column - 1, column)
    touching = [(row, column - 1), (row, column + 1)]
    for next_column in columns_in_next_rows:
        touching.append((row - 1, next_column))
        touching.append((row + 1, next_column))
    return touching


high_need_positions = {(hexagon['row'], hexagon['column']) for hexagon in high_need_hexagons}
collected = set()


def neighborhood_starting_at(position):
    neighborhood = []
    still_to_visit = [position]
    collected.add(position)
    while still_to_visit:
        current = still_to_visit.pop()
        neighborhood.append(current)
        row, column = current
        for neighbor in neighbors(row, column):
            if neighbor in high_need_positions and neighbor not in collected:
                collected.add(neighbor)
                still_to_visit.append(neighbor)
    return neighborhood


neighborhoods = []
for position in sorted(high_need_positions):
    if position not in collected:
        neighborhoods.append(neighborhood_starting_at(position))

neighborhood_sizes = sorted((count_of(neighborhood) for neighborhood in neighborhoods), reverse=True)

five_largest = ', '.join(str(size) for size in neighborhood_sizes[:5])
print(f'{count_of(neighborhoods)} neighborhoods')
print(f'the five largest hold {five_largest} hexagons')
print(f'{neighborhood_sizes.count(1)} are a single hexagon')

39 neighborhoods
the five largest hold 31, 28, 26, 17, 15 hexagons
10 are a single hexagon


**Check:** 39 neighborhoods, holding all 265 high-need hexagons between them.

In [6]:
assert count_of(neighborhoods) == 39
assert sum(neighborhood_sizes) == 265

## 3. How much the two lines matter

Both lines are judgment calls, so the Methodology counts the high-need hexagons under every combination of three bus lines (20, 25 and 30 minutes) and three car-ownership lines (more than 10, 15 and 20% of households without a car).

In [7]:
bus_lines_minutes = [20, 25, 30]
carless_lines_percent = [10, 15, 20]

hexagon_ids_by_combination = {
    (carless_line, bus_line): {hexagon['id'] for hexagon in lived_in_hexagons if is_high_need(hexagon, bus_line, carless_line)}
    for carless_line in carless_lines_percent
    for bus_line in bus_lines_minutes
}
sensitivity_table = [
    [count_of(hexagon_ids_by_combination[(carless_line, bus_line)]) for bus_line in bus_lines_minutes]
    for carless_line in carless_lines_percent
]

print('households without a car    bus 20 min or more   bus 25 min or more   bus 30 min or more')
for carless_line in carless_lines_percent:
    at_20_minutes = count_of(hexagon_ids_by_combination[(carless_line, 20)])
    at_25_minutes = count_of(hexagon_ids_by_combination[(carless_line, 25)])
    at_30_minutes = count_of(hexagon_ids_by_combination[(carless_line, 30)])
    print(f'more than {carless_line}%              {at_20_minutes:>5} hexagons       {at_25_minutes:>5} hexagons       {at_30_minutes:>5} hexagons')

households without a car    bus 20 min or more   bus 25 min or more   bus 30 min or more
more than 10%                565 hexagons         524 hexagons         466 hexagons
more than 15%                297 hexagons         265 hexagons         234 hexagons
more than 20%                182 hexagons         161 hexagons         140 hexagons


Shifting the bus threshold by 5 minutes either way changes the 265-hexagon count by up to **12%**, and **140 core hexagons** remain high-need across all nine combinations:

In [8]:
published_count = count_of(hexagon_ids_by_combination[(15, 25)])
looser_count, stricter_count = count_of(hexagon_ids_by_combination[(15, 20)]), count_of(hexagon_ids_by_combination[(15, 30)])
bus_line_swing_percent = max(looser_count - published_count, published_count - stricter_count) / published_count * 100

high_need_in_every_combination = [
    hexagon_id for hexagon_id in hexagon_ids_by_combination[(15, 25)]
    if all(hexagon_id in hexagon_ids for hexagon_ids in hexagon_ids_by_combination.values())
]

print(f'a bus line of 20 minutes adds {looser_count - published_count} hexagons; 30 minutes removes {published_count - stricter_count}')
print(f'the bigger change is {bus_line_swing_percent:.1f}% of the {published_count} published hexagons')
print(f'{count_of(high_need_in_every_combination)} of the {published_count} published hexagons are high-need in all nine combinations')

a bus line of 20 minutes adds 32 hexagons; 30 minutes removes 31
the bigger change is 12.1% of the 265 published hexagons
140 of the 265 published hexagons are high-need in all nine combinations


**Midday.** The published count uses bus trips leaving between 6 and 9 AM. Timing the same trips between 10 AM and 2 PM barely moves it.

In [9]:
midday_high_need = [hexagon for hexagon in lived_in_hexagons if is_high_need(hexagon, period='offPeak')]

print(f'{count_of(midday_high_need)} high-need hexagons at midday, against {published_count} at rush hour')

268 high-need hexagons at midday, against 265 at rush hour


**Check:** the nine counts in the table, a change of about 12% for five minutes either way, 140 hexagons high-need under all nine combinations, and 268 at midday.

In [10]:
assert sensitivity_table == [[565, 524, 466], [297, 265, 234], [182, 161, 140]]
assert round(bus_line_swing_percent) == 12
assert count_of(high_need_in_every_combination) == 140
assert count_of(midday_high_need) == 268

## 4. The USDA's map against the high-need hexagons

The story opens on the USDA's 35 low-income, low-access tracts and closes by laying the high-need hexagons over them. A hexagon is inside a tract when its center falls inside the tract's boundary, tested with the same east-pointing ray as in notebook 2.

In [11]:
usda_tract_layer = json.loads((snapshots_folder / 'Low_Income_Low_Access@2026-10-12.geojson').read_text())
usda_tract_boundaries = [
    feature['geometry']['coordinates']
    for feature in usda_tract_layer['features']
    if feature['properties']['COUNTYFP10'] == '113' and feature['properties']['LILATracts_1And10'] == 1
]


def boundary_edges(ring):
    return zip(ring, ring[-1:] + ring[:-1])


def ring_contains(point, ring):
    crossings = 0
    for (corner_lon, corner_lat), (previous_lon, previous_lat) in boundary_edges(ring):
        edge_spans_point_lat = (corner_lat > point['lat']) != (previous_lat > point['lat'])
        if not edge_spans_point_lat:
            continue
        edge_lon_at_point_lat = corner_lon + (point['lat'] - corner_lat) / (previous_lat - corner_lat) * (previous_lon - corner_lon)
        if point['lon'] < edge_lon_at_point_lat:
            crossings += 1
    return crossings % 2 == 1


def polygon_contains(point, rings):
    outer_ring = rings[0]
    holes = rings[1:]
    return ring_contains(point, outer_ring) and not any(ring_contains(point, hole) for hole in holes)


def in_a_usda_tract(hexagon):
    return any(polygon_contains(hexagon, rings) for rings in usda_tract_boundaries)


print(f'{count_of(usda_tract_boundaries)} tract boundaries')

35 tract boundaries


In [12]:
lived_in_inside_usda_tracts = [hexagon for hexagon in lived_in_hexagons if in_a_usda_tract(hexagon)]
high_need_inside_usda_tracts = [hexagon for hexagon in high_need_hexagons if in_a_usda_tract(hexagon)]
high_need_outside_usda_tracts = [hexagon for hexagon in high_need_hexagons if not in_a_usda_tract(hexagon)]


def carless_households_in(hexagons):
    return sum(hexagon['households'] * carless_percent_of(hexagon) / 100 for hexagon in hexagons)


inside, outside = high_need_inside_usda_tracts, high_need_outside_usda_tracts
print(f'high-need hexagons inside a USDA tract: {count_of(inside)}, with {carless_households_in(inside):,.0f} carless households')
print(f'high-need hexagons outside every tract: {count_of(outside)}, with {carless_households_in(outside):,.0f} carless households')

high-need hexagons inside a USDA tract: 141, with 3,561 carless households
high-need hexagons outside every tract: 124, with 2,124 carless households


Conversely, most lived-in hexagons inside the USDA tracts (480 of 621) are not high-need: most sit in block groups where at least 85% of households own a car, and every one of them is under a 15-minute drive from a supermarket:

In [13]:
usda_tract_hexagons_with_cars = [hexagon for hexagon in lived_in_inside_usda_tracts if carless_percent_of(hexagon) <= 15]
longest_drive_inside_usda_tracts = max(drive_minutes_to_supermarket(hexagon) for hexagon in lived_in_inside_usda_tracts)

print(f'lived-in hexagons inside the USDA tracts: {count_of(lived_in_inside_usda_tracts)}, of which {count_of(high_need_inside_usda_tracts)} are high-need')
print(f'in block groups where no more than 15% of households lack a car: {count_of(usda_tract_hexagons_with_cars)}')
print(f'longest drive to a supermarket from inside the tracts: {longest_drive_inside_usda_tracts:.1f} minutes')

lived-in hexagons inside the USDA tracts: 621, of which 141 are high-need
in block groups where no more than 15% of households lack a car: 447
longest drive to a supermarket from inside the tracts: 14.1 minutes


**Check:** 35 tracts; every high-need hexagon is either inside one or outside them all, 124 of them outside; and every lived-in hexagon inside the tracts is under 15 minutes from a supermarket by car.

In [14]:
assert count_of(usda_tract_boundaries) == 35
assert count_of(high_need_inside_usda_tracts) + count_of(high_need_outside_usda_tracts) == 265
assert count_of(high_need_outside_usda_tracts) == 124
assert longest_drive_inside_usda_tracts < 15

Nearly half of the county's high-need hexagons (**124 of 265**, home to **2,124 carless households**) lie outside the federal USDA tracts: places the federal measure does not flag, but where the bus to a supermarket takes 25 minutes or more, or no bus runs at all.

## Summary

| Check | Result | Where it appears on the site |
| :--- | :--- | :--- |
| High-need hexagons | **265**: 235 with a bus trip of 25 minutes or more, 30 with no bus trip | Story step 26, Methodology §3 |
| Households in them | **21,057**, about 5,700 without a car | Story step 26, the county finding |
| Outlined neighborhoods | **39** | The map's high-need outline |
| Nine combinations of the two lines | 565, 524, 466 / 297, 265, 234 / 182, 161, 140 | Methodology §3, *Cutoff Sensitivity* |
| Swing for 5 minutes either way | about **12%** | Methodology §3 |
| High-need under all nine combinations | **140** of 265 | Methodology §3 |
| At midday | **268** | Methodology §2, *Public Transit* |
| High-need hexagons outside every USDA tract | **124** of 265 | Story step 27 |

The cell below outputs these key figures as JSON, so they can be checked against the numbers the live site shows.

In [15]:
audited_figures = {
    'highNeedHexagons': count_of(high_need_hexagons),
    'highNeedHouseholds': round(high_need_households),
    'highNeedCarlessHouseholds': round(high_need_carless_households),
    'neighborhoods': count_of(neighborhoods),
    'sensitivityTable': sensitivity_table,
    'highNeedInEveryCombination': count_of(high_need_in_every_combination),
    'middayHighNeedHexagons': count_of(midday_high_need),
}

print(json.dumps(audited_figures, sort_keys=True))

{"highNeedCarlessHouseholds": 5684, "highNeedHexagons": 265, "highNeedHouseholds": 21057, "highNeedInEveryCombination": 140, "middayHighNeedHexagons": 268, "neighborhoods": 39, "sensitivityTable": [[565, 524, 466], [297, 265, 234], [182, 161, 140]]}


To rerun every audit notebook without Jupyter: `python3 notebooks/run_notebooks.py` from the project folder. It needs only Python 3.10 or newer.